In [67]:
import pandas as pd
import numpy as np
import random
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, ExtraTreesRegressor
from utils import *
from valid_models import valid_models

In [68]:
RANDOM_SEED = 1907
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Model Selection

In [69]:
"""Frequency encoding logic"""
def frequency_encode_train(df, cat_cols):
    df_encoded = df.copy()
    
    freq_values = {}
    for col in cat_cols:
        freq_map = df[col].value_counts(normalize=True)

        #this needed in order to not have data leakage when applying to test data, there will be classes that are not in the training dataset
        #due to the fact that the strings were corrupted and pre processing isnt perfect of course
        mean_freq = freq_map.mean()
        df_encoded[col] = df[col].map(freq_map).fillna(mean_freq)
        freq_values[col] = {
            "map": freq_map,
            "mean": mean_freq
        }

    return df_encoded, freq_values

In [70]:
def frequency_encode_test(df, cat_cols, freq_values):
    df_encoded = df.copy()

    for col in cat_cols:
        freq_map = freq_values[col]["map"]
        mean_freq = freq_values[col]["mean"]
        df_encoded[col] = df[col].map(freq_map).fillna(mean_freq)
    return df_encoded

In [71]:
def run_model(X, y, int_cols, float_cols, model_class, model_params=None,
              scaler=None, encoder=None, cat_cols=None, encoding_type="ohe"):

    X_processed = X.copy()

    if model_params is None:
        model_params = {}

    X_processed, fill_values = fill_nans(X_processed, int_cols, float_cols)
    X_processed, outlier_info = outliers_skews_train(X_processed)

    freq_values = None
    train_feature_cols = None
    freq_encoded_cols = []  # <-- track freq-encoded column names

    if cat_cols is not None and len(cat_cols) > 0:
        if encoding_type == "ohe" and encoder is not None:
            encoder.fit(X_processed[cat_cols])
            encoded_array = encoder.transform(X_processed[cat_cols])
            encoded_cols = encoder.get_feature_names_out(cat_cols)

            X_processed = pd.concat([
                X_processed.drop(columns=cat_cols).reset_index(drop=True),
                pd.DataFrame(encoded_array, columns=encoded_cols).reset_index(drop=True)
            ], axis=1)

        elif encoding_type == "freq":
            X_processed, freq_values = frequency_encode_train(X_processed, cat_cols)
            freq_encoded_cols = list(cat_cols)  # assuming freq encoding overwrites these cols

    train_feature_cols = X_processed.columns.tolist()

    if scaler is not None:
        # base numeric cols
        numeric_cols = [c for c in X_processed.columns if c in (int_cols + float_cols)]

        # if freq-encoding, also scale those (they're now numeric)
        if encoding_type == "freq":
            numeric_cols = list(dict.fromkeys(numeric_cols + freq_encoded_cols))  # de-dupe, keep order

        X_processed[numeric_cols] = scaler.fit_transform(X_processed[numeric_cols])

    model = model_class(**model_params)
    model.fit(X_processed, y)

    return model, scaler, encoder, fill_values, train_feature_cols, freq_values, outlier_info

In [72]:
def evaluate_model(X, y, model, int_cols, float_cols, fill_values,
                   scaler=None, encoder=None, cat_cols=None,
                   train_feature_cols=None, freq_values=None, outlier_info=None,
                   encoding_type="ohe", return_predictions=False):
    X_processed = X.copy()

    X_processed = fill_nans(X_processed, int_cols, float_cols, fill_values)

    if outlier_info is not None:
        X_processed = outliers_skews_test(X_processed, outlier_info)

    if cat_cols is not None and len(cat_cols) > 0:
        if encoding_type == "ohe" and encoder is not None:
            encoded_array = encoder.transform(X_processed[cat_cols])
            encoded_cols = encoder.get_feature_names_out(cat_cols)

            X_encoded = pd.concat([
                X_processed.drop(columns=cat_cols).reset_index(drop=True),
                pd.DataFrame(encoded_array, columns=encoded_cols).reset_index(drop=True)
            ], axis=1)

            # Align to training columns
            X_encoded = X_encoded.reindex(columns=train_feature_cols, fill_value=0)
            X_processed = X_encoded

        elif encoding_type == "freq":
            X_processed = frequency_encode_test(X_processed, cat_cols, freq_values)

    if scaler is not None:
        numeric_cols = [col for col in X_processed.columns if col in (int_cols + float_cols)]

        # also scale freq-encoded categorical cols (now numeric)
        if encoding_type == "freq" and cat_cols is not None:
            numeric_cols = list(dict.fromkeys(numeric_cols + list(cat_cols)))  # de-dupe

        X_processed[numeric_cols] = scaler.transform(X_processed[numeric_cols])

    preds = model.predict(X_processed)
    if return_predictions:
        return preds
    else:
        return model.score(X_processed, y)


In [73]:
def avg_score(method, X, y, int_cols, float_cols, model_class, model_params=None,
              scaler=None, encoder=None, cat_cols=None, encoding_type="ohe"):
    
    if model_params is None:
        model_params = {}
    
    mae_train, mae_val = [], []
    r2_train, r2_val = [], []

    for train_index, val_index in method.split(X, y):
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        trained_model, fitted_scaler, fitted_encoder, fill_values, train_feature_cols, freq_values, outlier_info = run_model(
            X_train, y_train,int_cols, float_cols, model_class=model_class, model_params=model_params,
            scaler=scaler, encoder=encoder,cat_cols=cat_cols, encoding_type=encoding_type)

    
        y_train_pred = evaluate_model(
            X_train, y_train, trained_model, int_cols, float_cols, fill_values,
            scaler=fitted_scaler, encoder=fitted_encoder, cat_cols=cat_cols, train_feature_cols=train_feature_cols,
            freq_values=freq_values, outlier_info=outlier_info, encoding_type=encoding_type, return_predictions=True
        )

        y_val_pred = evaluate_model(
            X_val, y_val, trained_model, int_cols, float_cols, fill_values,
            scaler=fitted_scaler, encoder=fitted_encoder, cat_cols=cat_cols, train_feature_cols=train_feature_cols,
            freq_values=freq_values, outlier_info=outlier_info, encoding_type=encoding_type, return_predictions=True
        )

        mae_train.append(mean_absolute_error(y_train, y_train_pred))
        mae_val.append(mean_absolute_error(y_val, y_val_pred))
        r2_train.append(r2_score(y_train, y_train_pred))
        r2_val.append(r2_score(y_val, y_val_pred))

    final_model, fitted_scaler, fitted_encoder, fill_values, train_feature_cols, freq_values, outlier_info = run_model(
        X, y, int_cols, float_cols, model_class=model_class, model_params=model_params, scaler=scaler, encoder=encoder,
        cat_cols=cat_cols, encoding_type=encoding_type)

    print(f"Average Train MAE: {np.mean(mae_train):.4f}")
    print(f"Average Val MAE:   {np.mean(mae_val):.4f}")
    print(f"Average Train R²:  {np.mean(r2_train):.4f}")
    print(f"Average Val R²:    {np.mean(r2_val):.4f}")

    return {
        "mae_train": mae_train,
        "mae_val": mae_val,
        "r2_train": r2_train,
        "r2_val": r2_val,
        "final_model": final_model,
        "fitted_scaler": fitted_scaler,
        "fitted_encoder": fitted_encoder,
        "fill_values": fill_values,
        "train_feature_cols": train_feature_cols,
        "freq_values": freq_values,
        "outlier_info": outlier_info
    }

In [74]:
def predict_test(X_test, model, int_cols, float_cols, fill_values,
                 scaler=None, encoder=None, cat_cols=None,
                 train_feature_cols=None, freq_values=None,
                 outlier_info=None, encoding_type="ohe"):
    X_processed = X_test.copy()

    # Fill missing values
    X_processed = fill_nans(X_processed, int_cols, float_cols, fill_values)

    # Handle outliers / skews
    if outlier_info is not None:
        X_processed = outliers_skews_test(X_processed, outlier_info)

    # Encode categorical features
    if cat_cols is not None and len(cat_cols) > 0:
        if encoding_type == "ohe" and encoder is not None:
            encoded_array = encoder.transform(X_processed[cat_cols])
            encoded_cols = encoder.get_feature_names_out(cat_cols)

            X_processed = pd.concat([
                X_processed.drop(columns=cat_cols).reset_index(drop=True),
                pd.DataFrame(encoded_array, columns=encoded_cols).reset_index(drop=True)
            ], axis=1)

            # Align to full training feature list
            X_processed = X_processed.reindex(columns=train_feature_cols, fill_value=0)

        elif encoding_type == "freq":
            X_processed = frequency_encode_test(X_processed, cat_cols, freq_values)

    # Scale features
    if scaler is not None:
        numeric_cols = [col for col in X_processed.columns if col in (int_cols + float_cols)]

        # also scale freq-encoded categorical cols (now numeric)
        if encoding_type == "freq" and cat_cols is not None:
            numeric_cols = list(dict.fromkeys(numeric_cols + list(cat_cols)))  # de-dupe

        X_processed[numeric_cols] = scaler.transform(X_processed[numeric_cols])

    return model.predict(X_processed)

In [75]:
def random_search_cv(
    X, y,
    int_cols, float_cols, cat_cols,
    model_class=RandomForestRegressor,
    base_params=None, # fixed params, e.g. {"random_state": 42, "n_jobs": -1}
    param_dist=None,
    n_iter=30,
    method=None,
    scaler=None,
    encoder=None,
    encoding_type="ohe",
    random_seed=42
):

    if param_dist is None:
        param_dist = {
            "n_estimators": [50, 75, 100, 150, 200, 300, 400, 500, 600],
            "max_depth": [5, 7, 10, 15, 20, 25, 30],
            "min_samples_split": [2, 4, 5, 10, 12, 15],
            "min_samples_leaf": [1, 2, 5, 7, 10],
            "max_features": ["sqrt", "log2", 0.8, 0.5, 0.7],
            "max_samples": [0.7, 0.8, 0.9],
            "bootstrap": [True]
        }

    #Randomly sample n_iter parameter combinations
    param_samples = [
        {key: random.choice(values) for key, values in param_dist.items()}
        for _ in range(n_iter)
    ]

    #Default to 7-fold CV if not specified
    if method is None:
        method = KFold(n_splits=7, shuffle=True, random_state=random_seed)

    cv_search_results = []

    print(f"\nRandomized Search with {method.get_n_splits()}-Fold CV (MAE Optimization)")
    print(f"Testing {n_iter} random parameter combinations\n")

    for i, sampled_params in enumerate(param_samples, start=1):
        fold_mae_train, fold_r2_train = [], []
        fold_mae_val, fold_r2_val = [], []

        base_params = base_params or {}
        model_params = {**base_params, **sampled_params}

        print(f"► Combination {i}/{n_iter}: {model_params}")

        for fold, (train_idx, val_idx) in enumerate(method.split(X, y), start=1):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]


            trained_model, fitted_scaler, fitted_encoder, fill_values, train_feature_cols, freq_values, outlier_info = run_model(
                X_train, y_train, int_cols, float_cols, model_class=model_class, model_params=model_params,
                scaler=scaler, encoder=encoder, cat_cols=cat_cols, encoding_type=encoding_type)

            print(f"[DEBUG] Combo {i}, Fold {fold}, model id = {id(trained_model)}")

            y_train_pred = evaluate_model(
                X_train, y_train, trained_model, int_cols, float_cols, fill_values,
                fitted_scaler, fitted_encoder, cat_cols, train_feature_cols,
                freq_values, outlier_info, encoding_type, return_predictions=True
            )

            y_val_pred = evaluate_model(
                X_val, y_val, trained_model, int_cols, float_cols, fill_values,
                fitted_scaler, fitted_encoder, cat_cols, train_feature_cols,
                freq_values, outlier_info, encoding_type, return_predictions=True
            )

            # Compute metrics
            mae_train = mean_absolute_error(y_train, y_train_pred)
            mae_val = mean_absolute_error(y_val, y_val_pred)
            r2_train = r2_score(y_train, y_train_pred)
            r2_val = r2_score(y_val, y_val_pred)

            fold_mae_train.append(mae_train)
            fold_mae_val.append(mae_val)
            fold_r2_train.append(r2_train)
            fold_r2_val.append(r2_val)

            print(f"   Fold {fold}: Train MAE = {mae_train:.4f}, Val MAE = {mae_val:.4f}, "
                  f"Train R² = {r2_train:.4f}, Val R² = {r2_val:.4f}")

        mean_mae_train = np.mean(fold_mae_train)
        mean_mae_val = np.mean(fold_mae_val)
        mean_r2_train = np.mean(fold_r2_train)
        mean_r2_val = np.mean(fold_r2_val)

        cv_search_results.append({
            **model_params,
            "mean_mae_train": mean_mae_train,
            "mean_mae_val": mean_mae_val,
            "mean_r2_train": mean_r2_train,
            "mean_r2_val": mean_r2_val
        })

        print(f"→ Avg Train MAE: {mean_mae_train:.4f} | Avg Val MAE: {mean_mae_val:.4f} | "
              f"Avg Train R²: {mean_r2_train:.4f} | Avg Val R²: {mean_r2_val:.4f}\n")

    #best parameter combo
    best_result = min(cv_search_results, key=lambda x: x["mean_mae_val"])
    best_params = {k: v for k, v in best_result.items()
                    if k not in ["mean_mae", "std_mae", "mean_r2", "std_r2"]}      # sem base_params

    print("\nBest Parameters (based on lowest validation MAE):")
    for k, v in best_params.items():
        print(f"  {k}: {v}")
    print(f"\nBest Train MAE: {best_result['mean_mae_train']:.4f}")
    print(f"Best Val MAE:   {best_result['mean_mae_val']:.4f}")
    print(f"Best Train R²:  {best_result['mean_r2_train']:.4f}")
    print(f"Best Val R²:    {best_result['mean_r2_val']:.4f}")

    return {
        "cv_results": cv_search_results,
        "best_result": best_result,
        "best_params": best_params
    }

## Model #1 : Linear Regression (Selected Features, OHE Encoding)

In [76]:
num_cols = ['year', "mileage_per_year", "power_efficiency", "engineSize", "mileage"]
cat_cols = ['Brand', 'transmission', 'model']
int_cols = ['year']
float_cols = ["mileage_per_year", "power_efficiency", "engineSize", "mileage"] 

In [77]:
df=pd.read_csv("train.csv")
dfcopy = df.copy()
df

,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
3,29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
4,10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75968,37194,Mercedes,C Class,2015.0,13498,Manual,14480.0,etrol,125.0,53.300000,2.0,78.0,0.000000,0.0
75969,6265,Audi,Q3,2013.0,12495,Semi-Auto,52134.0,Diesel,200.0,47.900000,2.0,38.0,2.000000,0.0
75970,54886,Toyota,Aygo,2017.0,8399,Automatic,11304.0,Petrol,145.0,67.000000,1.0,57.0,3.000000,0.0
75971,860,Audi,Q3,2015.0,12990,Manual,69072.0,iesel,125.0,60.100000,2.0,74.0,2.000000,0.0


In [78]:
X=clean_df(df, valid_models, cat_cols)
X, y = separar_y(X)
X

,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,mileage_per_year,power_efficiency
carID,,,,,,,,,,,,,
69512,VW,GOLF,4,SEMI-AUTO,28421.0,PETROL,NaN,11.417268,2.0,63.0,4,7105.25,0.175173
53000,Toyota,YARIS,1,MANUAL,4589.0,PETROL,145.0,47.900000,1.5,50.0,1,4589.0,0.031315
6366,Audi,Q2,1,SEMI-AUTO,3624.0,PETROL,145.0,40.900000,1.5,56.0,4,3624.0,0.036675
29021,Ford,FIESTA,2,MANUAL,9102.0,PETROL,145.0,65.700000,1.0,50.0,<NA>,4551.0,0.015221
10062,BMW,2SERIES,1,MANUAL,1000.0,PETROL,145.0,42.800000,1.5,97.0,3,1000.0,0.035047
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37194,Mercedes,CCLASS,5,MANUAL,14480.0,PETROL,125.0,53.300000,2.0,78.0,0,2896.0,0.037523
6265,Audi,Q3,7,SEMI-AUTO,52134.0,DIESEL,200.0,47.900000,2.0,38.0,2,7447.714286,0.041754
54886,Toyota,AYGO,3,AUTOMATIC,11304.0,PETROL,145.0,67.000000,1.0,57.0,3,3768.0,0.014925


In [79]:
drop_cols = ['paintQuality%', 'previousOwners', 'fuelType', "mpg", "tax"]
X = X.drop(columns=drop_cols)
X

,Brand,model,year,transmission,mileage,engineSize,mileage_per_year,power_efficiency
carID,,,,,,,,
69512,VW,GOLF,4,SEMI-AUTO,28421.0,2.0,7105.25,0.175173
53000,Toyota,YARIS,1,MANUAL,4589.0,1.5,4589.0,0.031315
6366,Audi,Q2,1,SEMI-AUTO,3624.0,1.5,3624.0,0.036675
29021,Ford,FIESTA,2,MANUAL,9102.0,1.0,4551.0,0.015221
10062,BMW,2SERIES,1,MANUAL,1000.0,1.5,1000.0,0.035047
...,...,...,...,...,...,...,...,...
37194,Mercedes,CCLASS,5,MANUAL,14480.0,2.0,2896.0,0.037523
6265,Audi,Q3,7,SEMI-AUTO,52134.0,2.0,7447.714286,0.041754
54886,Toyota,AYGO,3,AUTOMATIC,11304.0,1.0,3768.0,0.014925


In [80]:
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown="ignore")
scaler = StandardScaler()
kf = KFold(n_splits=7, shuffle=True, random_state=RANDOM_SEED)
results = avg_score(
    method=kf,
    X=X,
    y=y,
    int_cols=int_cols,
    float_cols=float_cols,
    model_class=LinearRegression,
    model_params=None,
    scaler=scaler,
    encoder=ohe,
    cat_cols=cat_cols,
    encoding_type="ohe"
)

c:\Users\franc\Desktop\ml2526\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\franc\Desktop\ml2526\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\franc\Desktop\ml2526\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\franc\Desktop\ml2526\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\franc\Desktop\ml2526\.venv\Lib\site-packages\sk

Average Train MAE: 2474.8935
Average Val MAE:   2486.4803
Average Train R²:  0.8290
Average Val R²:    0.8267


In [81]:
model = results["final_model"]
scaler = results["fitted_scaler"]
encoder = results["fitted_encoder"]
fill_values = results["fill_values"]
train_feature_cols = results["train_feature_cols"]
outlier_info = results["outlier_info"]

In [82]:
import joblib

bundle = {
    "model": results["final_model"],
    "scaler": results["fitted_scaler"],
    "encoder": results["fitted_encoder"],
    "fill_values": results["fill_values"],
    "train_feature_cols": results["train_feature_cols"],
    "outlier_info": results["outlier_info"],
    "int_cols": int_cols,
    "float_cols": float_cols,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "encoding_type": "ohe"
}

joblib.dump(bundle, "linear_model_bundle.joblib")

['linear_model_bundle.joblib']

In [83]:
test_df = pd.read_csv("test.csv")
test_copy = test_df.copy()
X_test = clean_df(test_copy, valid_models, cat_cols)
X_test = X_test.drop(columns=drop_cols)
XcarID=X_test.copy()
X_test

,Brand,model,year,transmission,mileage,engineSize,mileage_per_year,power_efficiency
carID,,,,,,,,
89856,Hyundai,I30,<NA>,AUTOMATIC,30700.0,1.6,<NA>,0.038554
106581,VW,TIGUAN,3,SEMI-AUTO,NaN,2.0,<NA>,0.052356
80886,BMW,2SERIES,4,AUTOMATIC,36792.0,1.5,9198.0,0.029183
100174,Opel,GRANDLANDX,1,MANUAL,5533.0,1.2,5533.0,0.027211
81376,BMW,1SERIES,1,SEMI-AUTO,9058.0,2.0,9058.0,0.038911
...,...,...,...,...,...,...,...,...
105775,VW,TIGUAN,3,MANUAL,27575.0,1.4,9191.666667,0.030238
81363,BMW,X2,0,AUTOMATIC,1980.0,2.0,1980.0,0.058824
76833,Audi,Q5,1,SEMI-AUTO,8297.0,2.0,8297.0,0.052356


In [84]:
y_pred = predict_test(
    X_test,
    model=model,
    int_cols=int_cols,
    float_cols=float_cols,
    fill_values=fill_values,
    scaler=scaler,
    encoder=encoder,
    cat_cols=cat_cols,
    train_feature_cols=train_feature_cols,
    outlier_info=outlier_info
)

c:\Users\franc\Desktop\ml2526\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [85]:
submission = pd.DataFrame({
    "carID": XcarID.index,  
    "price": y_pred
})

# Check structure
display(submission.head())

# --- Save to CSV (with header, no index) ---
submission.to_csv("kaggle/model1.csv", index=False)

,carID,price
0,89856,14493.583648
1,106581,23423.372746
2,80886,13362.685857
3,100174,17696.867027
4,81376,22105.016726


## Model #2 : Random Forests (Selected Features, OHE Encoding)

In [86]:
num_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year', 'power_efficiency']
cat_cols = ['Brand', 'model', 'transmission']
int_cols = ['year']
float_cols = ['mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year', 'power_efficiency']

drop_cols = ['paintQuality%', 'previousOwners', 'fuelType']

In [87]:
df=pd.read_csv("train.csv")
dfcopy = df.copy()
X=clean_df(df, valid_models, cat_cols)
X, y = separar_y(X)
X = X.drop(columns=drop_cols)
X

,Brand,model,year,transmission,mileage,tax,mpg,engineSize,mileage_per_year,power_efficiency
carID,,,,,,,,,,
69512,VW,GOLF,4,SEMI-AUTO,28421.0,NaN,11.417268,2.0,7105.25,0.175173
53000,Toyota,YARIS,1,MANUAL,4589.0,145.0,47.900000,1.5,4589.0,0.031315
6366,Audi,Q2,1,SEMI-AUTO,3624.0,145.0,40.900000,1.5,3624.0,0.036675
29021,Ford,FIESTA,2,MANUAL,9102.0,145.0,65.700000,1.0,4551.0,0.015221
10062,BMW,2SERIES,1,MANUAL,1000.0,145.0,42.800000,1.5,1000.0,0.035047
...,...,...,...,...,...,...,...,...,...,...
37194,Mercedes,CCLASS,5,MANUAL,14480.0,125.0,53.300000,2.0,2896.0,0.037523
6265,Audi,Q3,7,SEMI-AUTO,52134.0,200.0,47.900000,2.0,7447.714286,0.041754
54886,Toyota,AYGO,3,AUTOMATIC,11304.0,145.0,67.000000,1.0,3768.0,0.014925


In [88]:
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown="ignore")
scaler=None

In [89]:
"""results = random_search_cv(
    X, y,
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    base_params={"random_state": RANDOM_SEED, "n_jobs": -1},
    encoder=ohe,
    encoding_type="ohe",
    random_seed=RANDOM_SEED)
    
    
    Best Train MAE: 769.9891
Best Val MAE:   1374.8723
Best Train R²:  0.9812
Best Val R²:    0.9378
"""


'results = random_search_cv(\n    X, y,\n    int_cols=int_cols,\n    float_cols=float_cols,\n    cat_cols=cat_cols,\n    base_params={"random_state": RANDOM_SEED, "n_jobs": -1},\n    encoder=ohe,\n    encoding_type="ohe",\n    random_seed=RANDOM_SEED)\n\n\n    Best Train MAE: 769.9891\nBest Val MAE:   1374.8723\nBest Train R²:  0.9812\nBest Val R²:    0.9378\n'

In [90]:
best_params = {"random_state": RANDOM_SEED,
  "n_jobs": -1,
  "n_estimators": 75,
  "max_depth": 30,
  "min_samples_split": 4,
  "min_samples_leaf": 1,
  "max_features": 0.7,
  "max_samples": 0.8,
  "bootstrap": True}

In [ ]:
rf_params = best_params
rf_model=RandomForestRegressor
final_model, fitted_scaler, fitted_encoder, fill_values, train_feature_cols, freq_values, outlier_info = run_model(
    X, y, int_cols, float_cols, model_class=rf_model, model_params=rf_params, scaler=None, encoder=ohe,
    cat_cols=cat_cols,
    encoding_type="ohe"
)

In [ ]:
rf_bundle = {
    "model": final_model,              # trained RandomForestRegressor
    "scaler": fitted_scaler,            # None (explicitly saved)
    "encoder": fitted_encoder,          # fitted OneHotEncoder
    "fill_values": fill_values,
    "train_feature_cols": train_feature_cols,
    "freq_values": freq_values,         # None for OHE
    "outlier_info": outlier_info,
    "int_cols": int_cols,
    "float_cols": float_cols,
    "cat_cols": cat_cols,
    "encoding_type": "ohe"
}

joblib.dump(rf_bundle, "random_forest_ohe.joblib")

['random_forest_ohe.joblib']

In [ ]:
test_df = pd.read_csv("test.csv")
test_copy = test_df.copy()
X_test = clean_df(test_copy, valid_models, cat_cols)
X_test = X_test.drop(columns=drop_cols)
XcarID=X_test.copy()

In [ ]:
y_pred = predict_test(
    X_test,
    model=final_model,
    int_cols=int_cols,
    float_cols=float_cols,
    fill_values=fill_values,
    scaler=fitted_scaler,
    encoder=fitted_encoder,
    cat_cols=cat_cols,
    train_feature_cols=train_feature_cols,
    freq_values=freq_values,
    encoding_type="ohe",
    outlier_info=outlier_info
)

c:\Users\franc\Desktop\ml2526\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [ ]:
submission = pd.DataFrame({
    "carID": XcarID.index,  
    "price": y_pred
})

# Check structure
display(submission.head())

# --- Save to CSV (with header, no index) ---
submission.to_csv("kaggle/model2.csv", index=False)

,carID,price
0,89856,13849.381509
1,106581,23928.334791
2,80886,13570.220889
3,100174,16541.689833
4,81376,25376.109409


## Model #3 : MLPRegressor (Selected Features, OHE Encoding)

In [ ]:
num_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year', 'power_efficiency']
cat_cols = ['Brand', 'model', 'transmission']
int_cols = ['year']
float_cols = ['mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year', 'power_efficiency']

In [ ]:
drop_cols = ['paintQuality%', 'previousOwners', 'fuelType']
df=pd.read_csv("train.csv")
dfcopy = df.copy()
X=clean_df(df, valid_models, cat_cols)
X, y = separar_y(X)
X = X.drop(columns=drop_cols)
X

,Brand,model,year,transmission,mileage,tax,mpg,engineSize,mileage_per_year,power_efficiency
carID,,,,,,,,,,
69512,VW,GOLF,4,SEMI-AUTO,28421.0,NaN,11.417268,2.0,7105.25,0.175173
53000,Toyota,YARIS,1,MANUAL,4589.0,145.0,47.900000,1.5,4589.0,0.031315
6366,Audi,Q2,1,SEMI-AUTO,3624.0,145.0,40.900000,1.5,3624.0,0.036675
29021,Ford,FIESTA,2,MANUAL,9102.0,145.0,65.700000,1.0,4551.0,0.015221
10062,BMW,2SERIES,1,MANUAL,1000.0,145.0,42.800000,1.5,1000.0,0.035047
...,...,...,...,...,...,...,...,...,...,...
37194,Mercedes,CCLASS,5,MANUAL,14480.0,125.0,53.300000,2.0,2896.0,0.037523
6265,Audi,Q3,7,SEMI-AUTO,52134.0,200.0,47.900000,2.0,7447.714286,0.041754
54886,Toyota,AYGO,3,AUTOMATIC,11304.0,145.0,67.000000,1.0,3768.0,0.014925


In [ ]:
y = np.log1p(y) #log transform target

In [ ]:
base_params = {
    "solver": "adam",
    "early_stopping": True,
    "random_state": RANDOM_SEED,
    "validation_fraction": 0.1,   # 10% of TRAIN fold used for internal ES
    "n_iter_no_change": 7,       # early stopping patience
}

In [ ]:
param_dist = {
    "hidden_layer_sizes": [
        (7,), (8,), (9,), (10,),
        
        (6, 4),
        (7, 5),
        (8, 6),
        (9, 5),
        (10, 6),

        (5, 3, 2),
        (6, 4, 2),
        (7, 5, 3),
        (8, 6, 4),
        (8, 5, 3),
        (9, 7, 5),
        (10, 8, 6),

        (10, 8, 6, 4),
        (9, 7, 5, 3),
        (8, 6, 4, 2),
        (7, 5, 3, 2),
    ],

    "activation": ["relu", "identity"],

    "learning_rate_init": [
        5e-4,
        1e-3,
        2e-3,
        3e-3,
        5e-3,
    ],

    "batch_size": [128, 256, 512],

    "alpha": [
        1e-4,
        3e-4,
        1e-3,
        3e-3,
        2e-3,
        5e-3,
        6e-3,
        1e-2,
    ],
}

In [ ]:
scaler = MinMaxScaler()
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

"""results = random_search_cv(
    X, y,
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    model_class=MLPRegressor,
    base_params=base_params,
    param_dist=param_dist,
    n_iter=30,
    method=None,
    scaler=scaler,
    encoder=encoder,
    encoding_type="ohe",
    random_seed=RANDOM_SEED,
)

Best Train MAE: 0.1080
Best Val MAE:   0.1094
Best Train R²:  0.9185
Best Val R²:    0.9144
"""

'results = random_search_cv(\n    X, y,\n    int_cols=int_cols,\n    float_cols=float_cols,\n    cat_cols=cat_cols,\n    model_class=MLPRegressor,\n    base_params=base_params,\n    param_dist=param_dist,\n    n_iter=30,\n    method=None,\n    scaler=scaler,\n    encoder=encoder,\n    encoding_type="ohe",\n    random_seed=RANDOM_SEED,\n)\n\nBest Train MAE: 0.1080\nBest Val MAE:   0.1094\nBest Train R²:  0.9185\nBest Val R²:    0.9144\n'

In [ ]:
#EXP TRANSFORM PRICE AFTER

In [ ]:
best_params = {"solver": "adam",
  "early_stopping": True,
  "random_state": 1907,
  "validation_fraction": 0.1,
  "n_iter_no_change": 7,
  "hidden_layer_sizes": (9, 7, 5),
  "activation": "relu",
  "learning_rate_init": 0.0005,
  "batch_size": 512,
  "alpha": 0.0003}

In [ ]:
mlpr_params = best_params
mlpr_model=MLPRegressor
final_model, fitted_scaler, fitted_encoder, fill_values, train_feature_cols, freq_values, outlier_info = run_model(
    X, y, int_cols, float_cols, model_class=mlpr_model, model_params=mlpr_params, scaler=scaler, encoder=encoder,
    cat_cols=cat_cols,
    encoding_type="ohe"
)

In [ ]:
rf_bundle = {
    "model": final_model,   
    "scaler": fitted_scaler,
    "encoder": fitted_encoder, 
    "fill_values": fill_values,
    "train_feature_cols": train_feature_cols,
    "freq_values": freq_values,
    "outlier_info": outlier_info,
    "int_cols": int_cols,
    "float_cols": float_cols,
    "cat_cols": cat_cols,
    "encoding_type": "ohe",
    "log_transform_target": True
}

joblib.dump(rf_bundle, "mlpregressor_ohe.joblib")

['mlpregressor_ohe.joblib']

In [ ]:
test_df = pd.read_csv("test.csv")
test_copy = test_df.copy()
X_test = clean_df(test_copy, valid_models, cat_cols)
X_test = X_test.drop(columns=drop_cols)
XcarID=X_test.copy()

In [ ]:
y_pred = predict_test(
    X_test,
    model=final_model,
    int_cols=int_cols,
    float_cols=float_cols,
    fill_values=fill_values,
    scaler=fitted_scaler,
    encoder=fitted_encoder,
    cat_cols=cat_cols,
    train_feature_cols=train_feature_cols,
    freq_values=freq_values,
    encoding_type="ohe",
    outlier_info=outlier_info
)

In [ ]:
submission = pd.DataFrame({
    "carID": XcarID.index,  
    "price": np.expm1(y_pred)
})

# Check structure
display(submission.head())

# --- Save to CSV (with header, no index) ---
submission.to_csv("kaggle/model3.csv", index=False)

,carID,price
0,89856,13265.860313
1,106581,21484.855498
2,80886,13247.929860
3,100174,17716.574095
4,81376,22872.129606


## Model #4 : HistGradientBoostingRegressor (Selected Features, OHE Encoding, MinMaxScaler)

In [ ]:
num_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year', 'power_efficiency']
cat_cols = ['Brand', 'model', 'transmission']
int_cols = ['year']
float_cols = ['mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year', 'power_efficiency']

In [ ]:
drop_cols = ['paintQuality%', 'previousOwners', 'fuelType']
df=pd.read_csv("train.csv")
dfcopy = df.copy()
X=clean_df(df, valid_models, cat_cols)
X, y = separar_y(X)
X = X.drop(columns=drop_cols)
X

,Brand,model,year,transmission,mileage,tax,mpg,engineSize,mileage_per_year,power_efficiency
carID,,,,,,,,,,
69512,VW,GOLF,4,SEMI-AUTO,28421.0,NaN,11.417268,2.0,7105.25,0.175173
53000,Toyota,YARIS,1,MANUAL,4589.0,145.0,47.900000,1.5,4589.0,0.031315
6366,Audi,Q2,1,SEMI-AUTO,3624.0,145.0,40.900000,1.5,3624.0,0.036675
29021,Ford,FIESTA,2,MANUAL,9102.0,145.0,65.700000,1.0,4551.0,0.015221
10062,BMW,2SERIES,1,MANUAL,1000.0,145.0,42.800000,1.5,1000.0,0.035047
...,...,...,...,...,...,...,...,...,...,...
37194,Mercedes,CCLASS,5,MANUAL,14480.0,125.0,53.300000,2.0,2896.0,0.037523
6265,Audi,Q3,7,SEMI-AUTO,52134.0,200.0,47.900000,2.0,7447.714286,0.041754
54886,Toyota,AYGO,3,AUTOMATIC,11304.0,145.0,67.000000,1.0,3768.0,0.014925


In [ ]:
base_params = {
    "early_stopping": True,
    "validation_fraction": 0.1,
    "n_iter_no_change": 7,
}

In [ ]:
param_dist = {
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.07, 0.1],
    "max_iter":      [200, 400, 800, 1200],

    "max_depth":     [None, 3, 4, 5, 6, 8],
    "max_leaf_nodes":[15, 31, 63, 127],

    "min_samples_leaf": [10, 20, 30, 50, 80, 120],
    "l2_regularization":[0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0],

    "max_bins": [64, 128, 255]}

In [ ]:
scaler = MinMaxScaler()
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

In [ ]:
"""results = random_search_cv(
    X, y,
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    model_class=HistGradientBoostingRegressor,
    base_params=base_params,
    param_dist=param_dist,
    n_iter=30,
    method=None,
    scaler=scaler,
    encoder=encoder,
    encoding_type="ohe",
    random_seed=RANDOM_SEED,
)


Best Train MAE: 1088.7667
Best Val MAE:   1326.7864
Best Train R²:  0.9693
Best Val R²:    0.9435"""

'results = random_search_cv(\n    X, y,\n    int_cols=int_cols,\n    float_cols=float_cols,\n    cat_cols=cat_cols,\n    model_class=HistGradientBoostingRegressor,\n    base_params=base_params,\n    param_dist=param_dist,\n    n_iter=30,\n    method=None,\n    scaler=scaler,\n    encoder=encoder,\n    encoding_type="ohe",\n    random_seed=RANDOM_SEED,\n)\n\n\nBest Train MAE: 1088.7667\nBest Val MAE:   1326.7864\nBest Train R²:  0.9693\nBest Val R²:    0.9435'

In [ ]:
best_params = {"early_stopping": True,
  "validation_fraction": 0.1,
  "n_iter_no_change": 7,
  "learning_rate": 0.07,
  "max_iter": 400,
  "max_depth": None,
  "max_leaf_nodes": 127,
  "min_samples_leaf": 10,
  "l2_regularization": 0.01,
  "max_bins": 255}

In [ ]:
hgb_params = best_params
hgb_model=HistGradientBoostingRegressor
final_model, fitted_scaler, fitted_encoder, fill_values, train_feature_cols, freq_values, outlier_info = run_model(
    X, y, int_cols, float_cols, model_class=hgb_model, model_params=hgb_params, scaler=scaler, encoder=encoder,
    cat_cols=cat_cols,
    encoding_type="ohe"
)

In [ ]:
rf_bundle = {
    "model": final_model,   
    "scaler": fitted_scaler,
    "encoder": fitted_encoder, 
    "fill_values": fill_values,
    "train_feature_cols": train_feature_cols,
    "freq_values": freq_values,
    "outlier_info": outlier_info,
    "int_cols": int_cols,
    "float_cols": float_cols,
    "cat_cols": cat_cols,
    "encoding_type": "ohe"
}

joblib.dump(rf_bundle, "hgb_ohe.joblib")

['hgb_ohe.joblib']

In [ ]:
test_df = pd.read_csv("test.csv")
test_copy = test_df.copy()
X_test = clean_df(test_copy, valid_models, cat_cols)
X_test = X_test.drop(columns=drop_cols)
XcarID=X_test.copy()

In [ ]:
y_pred = predict_test(
    X_test,
    model=final_model,
    int_cols=int_cols,
    float_cols=float_cols,
    fill_values=fill_values,
    scaler=fitted_scaler,
    encoder=fitted_encoder,
    cat_cols=cat_cols,
    train_feature_cols=train_feature_cols,
    freq_values=freq_values,
    encoding_type="ohe",
    outlier_info=outlier_info
)

In [ ]:
submission = pd.DataFrame({
    "carID": XcarID.index,  
    "price": y_pred
})

# Check structure
display(submission.head())

# --- Save to CSV (with header, no index) ---
submission.to_csv("kaggle/model4.csv", index=False)

,carID,price
0,89856,10362.831688
1,106581,24188.473238
2,80886,13538.960829
3,100174,17646.865411
4,81376,23420.738659


## Model #5 : ExtraTrees (Selected Features, OHE Encoding)

In [ ]:
num_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year', 'power_efficiency']
cat_cols = ['Brand', 'model', 'transmission']
int_cols = ['year']
float_cols = ['mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year', 'power_efficiency']
drop_cols = ['paintQuality%', 'previousOwners', 'fuelType']

In [ ]:
df=pd.read_csv("train.csv")
dfcopy = df.copy()
X=clean_df(df, valid_models, cat_cols)
X, y = separar_y(X)
X = X.drop(columns=drop_cols)
X

,Brand,model,year,transmission,mileage,tax,mpg,engineSize,mileage_per_year,power_efficiency
carID,,,,,,,,,,
69512,VW,GOLF,4,SEMI-AUTO,28421.0,NaN,11.417268,2.0,7105.25,0.175173
53000,Toyota,YARIS,1,MANUAL,4589.0,145.0,47.900000,1.5,4589.0,0.031315
6366,Audi,Q2,1,SEMI-AUTO,3624.0,145.0,40.900000,1.5,3624.0,0.036675
29021,Ford,FIESTA,2,MANUAL,9102.0,145.0,65.700000,1.0,4551.0,0.015221
10062,BMW,2SERIES,1,MANUAL,1000.0,145.0,42.800000,1.5,1000.0,0.035047
...,...,...,...,...,...,...,...,...,...,...
37194,Mercedes,CCLASS,5,MANUAL,14480.0,125.0,53.300000,2.0,2896.0,0.037523
6265,Audi,Q3,7,SEMI-AUTO,52134.0,200.0,47.900000,2.0,7447.714286,0.041754
54886,Toyota,AYGO,3,AUTOMATIC,11304.0,145.0,67.000000,1.0,3768.0,0.014925


In [ ]:
base_params = {
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
}

In [ ]:
param_dist = {
    "n_estimators": [200, 300, 400, 600, 800, 1000],

    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10, 20],

    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7, 1.0],

    "bootstrap": [False]
}

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

In [ ]:
"""results = random_search_cv(
    X, y,
    int_cols=int_cols,
    float_cols=float_cols,
    cat_cols=cat_cols,
    model_class=ExtraTreesRegressor,
    base_params=base_params,
    param_dist=param_dist,
    n_iter=30,
    method=None,
    scaler=None,
    encoder=encoder,
    encoding_type="ohe",
    random_seed=RANDOM_SEED,
)


Best Train MAE: 754.3044
Best Val MAE:   1299.7174
Best Train R²:  0.9818
Best Val R²:    0.9435"""

'results = random_search_cv(\n    X, y,\n    int_cols=int_cols,\n    float_cols=float_cols,\n    cat_cols=cat_cols,\n    model_class=ExtraTreesRegressor,\n    base_params=base_params,\n    param_dist=param_dist,\n    n_iter=30,\n    method=None,\n    scaler=None,\n    encoder=encoder,\n    encoding_type="ohe",\n    random_seed=RANDOM_SEED,\n)\n\n\nBest Train MAE: 754.3044\nBest Val MAE:   1299.7174\nBest Train R²:  0.9818\nBest Val R²:    0.9435'

In [ ]:
best_params = {
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "n_estimators": 1000,
    "max_depth": 40,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "max_features": 0.7,
    "bootstrap": False
}

In [ ]:
et_params = best_params
et_model=ExtraTreesRegressor
final_model, fitted_scaler, fitted_encoder, fill_values, train_feature_cols, freq_values, outlier_info = run_model(
    X, y, int_cols, float_cols, model_class=et_model, model_params=et_params, scaler=None, encoder=encoder,
    cat_cols=cat_cols,
    encoding_type="ohe"
)

In [ ]:
rf_bundle = {
    "model": final_model,   
    "scaler": fitted_scaler,
    "encoder": fitted_encoder, 
    "fill_values": fill_values,
    "train_feature_cols": train_feature_cols,
    "freq_values": freq_values,
    "outlier_info": outlier_info,
    "int_cols": int_cols,
    "float_cols": float_cols,
    "cat_cols": cat_cols,
    "encoding_type": "ohe"
}

joblib.dump(rf_bundle, "et_ohe.joblib")

['et_ohe.joblib']

In [ ]:
test_df = pd.read_csv("test.csv")
test_copy = test_df.copy()
X_test = clean_df(test_copy, valid_models, cat_cols)
X_test = X_test.drop(columns=drop_cols)
XcarID=X_test.copy()

In [ ]:
y_pred = predict_test(
    X_test,
    model=final_model,
    int_cols=int_cols,
    float_cols=float_cols,
    fill_values=fill_values,
    scaler=fitted_scaler,
    encoder=fitted_encoder,
    cat_cols=cat_cols,
    train_feature_cols=train_feature_cols,
    freq_values=freq_values,
    encoding_type="ohe",
    outlier_info=outlier_info
)

In [ ]:
submission = pd.DataFrame({
    "carID": XcarID.index,  
    "price": y_pred
})

# Check structure
display(submission.head())

# --- Save to CSV (with header, no index) ---
submission.to_csv("kaggle/model5.csv", index=False)

,carID,price
0,89856,11678.017093
1,106581,23842.836948
2,80886,14057.870606
3,100174,16454.661525
4,81376,23144.013380
